# 🦅 PREDATOR Analytics v63.2-ELITE: Kaggle Native Node
**Архітектура**: FastAPI Native + SQLite/Redis-lite + zrok tunnel

> ⚠️ K3s/Docker ЗАБОРОНЕНО в Kaggle (kernel restriction). Використовуємо нативні процеси.

In [ ]:
# === КЛІТИНКА 1: Встановлення залежностей ===
import subprocess, sys

packages = [
    'fastapi', 'uvicorn[standard]', 'psutil', 'httpx',
    'python-jose[cryptography]', 'passlib[bcrypt]',
    'sqlalchemy', 'aiosqlite', 'redis', 'orjson'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('✅ Залежності встановлено')

In [ ]:
# === КЛІТИНКА 2: Завантаження актуального zrok ===
import subprocess, os, json, urllib.request

ZROK_DIR = '/kaggle/working'
ZROK_BIN = f'{ZROK_DIR}/zrok'

if not os.path.exists(ZROK_BIN):
    print('📦 Визначення актуальної версії zrok...')
    # Отримуємо останній реліз динамічно
    try:
        with urllib.request.urlopen(
            'https://api.github.com/repos/openziti/zrok/releases/latest',
            timeout=10
        ) as r:
            rel = json.loads(r.read())
            tag = rel['tag_name']  # наприклад v0.4.45
    except Exception:
        tag = 'v1.0.0'  # fallback
    
    ver = tag.lstrip('v')
    url = f'https://github.com/openziti/zrok/releases/download/{tag}/zrok_{ver}_linux_amd64.tar.gz'
    print(f'🔽 Завантаження zrok {tag} з {url}')
    subprocess.run(f'cd {ZROK_DIR} && wget -q "{url}" -O zrok.tar.gz && tar -xzf zrok.tar.gz && chmod +x zrok', shell=True)
    print(f'✅ zrok {tag} встановлено')
else:
    print(f'✅ zrok вже є: {ZROK_BIN}')

# Перевірка версії
r = subprocess.run([ZROK_BIN, 'version'], capture_output=True, text=True)
print(f'zrok version: {r.stdout.strip() or r.stderr.strip()}')

In [ ]:
# === КЛІТИНКА 3: Активація zrok токену ===
ZROK_TOKEN = '1eeje4um7yvA'  # ← ВАШ ТОКЕН

# Скидаємо попередній стан якщо є
subprocess.run([ZROK_BIN, 'disable'], capture_output=True)

r = subprocess.run([ZROK_BIN, 'enable', ZROK_TOKEN], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ zrok активовано')
else:
    print(f'⚠️ zrok enable: {r.stdout} {r.stderr}')
    print('   → Спробуємо запустити тунель без enable (може вже активований)')

In [ ]:
# === КЛІТИНКА 4: Запис PREDATOR FastAPI Backend ===

backend_code = '''
import os, time, threading, psutil
from datetime import datetime, timezone
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="PREDATOR Analytics Kaggle Node", version="63.2-ELITE")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# OODA стан
class OodaState:
    running = False
    phase = "IDLE"
    cycles = 0
    logs = []
    _lock = threading.Lock()

def _ooda_worker():
    phases = ["OBSERVE", "ORIENT", "DECIDE", "ACT"]
    descs = {
        "OBSERVE": "Сканування транзакційних потоків...",
        "ORIENT":  "Аналіз відхилень від Gold Pattern v5.0...",
        "DECIDE":  "Формування стратегії блокування...",
        "ACT":     "Впровадження правил у Risk Engine...",
    }
    while OodaState.running:
        for p in phases:
            if not OodaState.running:
                break
            OodaState.phase = p
            ts = datetime.now().strftime("%H:%M:%S")
            with OodaState._lock:
                OodaState.logs.append(f"[{ts}] {p}: {descs[p]}")
                if len(OodaState.logs) > 40:
                    OodaState.logs.pop(0)
            time.sleep(8)
        OodaState.cycles += 1

# Ендпоїнти
@app.get("/api/v1/health")
async def health():
    mem = psutil.virtual_memory()
    return {
        "status": "ONLINE",
        "mode": "KAGGLE_NATIVE",
        "node": "KAGGLE_RESERVE",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "ram_used_gb": round(mem.used / 1024**3, 2),
        "ram_total_gb": round(mem.total / 1024**3, 2),
        "ram_percent": mem.percent,
        "cpu_percent": psutil.cpu_percent(interval=0.1),
    }

@app.get("/api/v1/factory/ooda")
async def get_ooda():
    return {
        "is_running": OodaState.running,
        "current_phase": OodaState.phase,
        "cycles": OodaState.cycles,
        "logs": OodaState.logs,
    }

@app.post("/api/v1/factory/infinite/start")
async def start_ooda():
    if not OodaState.running:
        OodaState.running = True
        threading.Thread(target=_ooda_worker, daemon=True).start()
    return {"status": "started"}

@app.post("/api/v1/factory/infinite/stop")
async def stop_ooda():
    OodaState.running = False
    OodaState.phase = "IDLE"
    return {"status": "stopped"}

@app.get("/api/v1/risk/company/{ueid}")
async def get_risk(ueid: str):
    return {
        "ueid": ueid,
        "score": 74.2,
        "level": "CRITICAL",
        "layers": {"structural": 88, "behavioral": 62, "sanctions": 95, "aml": 45},
        "explanation": "Виявлено аномальну активність (KAGGLE_NODE OODA).",
    }

@app.get("/api/v1/tornado/stats")
async def get_tornado():
    return {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "modules": [
            {"id": "forecast", "accuracy": 94.2, "status": "ACTIVE"},
            {"id": "market",   "entities": 1200,  "status": "LIVE"},
            {"id": "graph",    "nodes": 45000,    "status": "ACTIVE"},
            {"id": "diligence","flags": 12,        "status": "ACTIVE"},
            {"id": "anomaly",  "tps": 847,         "status": "LIVE"},
            {"id": "scenario", "active": 3,        "status": "WAR_ROOM"},
        ],
    }

@app.get("/api/v1/osint/diligence/{ueid}")
async def get_diligence(ueid: str):
    return {
        "ueid": ueid,
        "name": "ТОВ ТЕСТОВА КОМПАНІЯ",
        "red_flags": ["SUDDEN_DIRECTOR_CHANGE", "HIGH_DEBT_RATIO"],
        "summary": "Високий ризик фіктивності. Рекомендовано поглиблений аудит.",
    }

@app.get("/api/v1/analytics/summary")
async def analytics_summary():
    return {
        "total_companies": 125430,
        "high_risk": 3241,
        "sanctions_hit": 87,
        "daily_transactions": 84729,
        "node": "KAGGLE_RESERVE",
    }

# Автостарт OODA
OodaState.running = True
threading.Thread(target=_ooda_worker, daemon=True).start()
print("✅ PREDATOR Kaggle Backend готовий")
'''

with open('/kaggle/working/predator_app.py', 'w') as f:
    f.write(backend_code)

print('✅ Backend записано: /kaggle/working/predator_app.py')

In [ ]:
# === КЛІТИНКА 5: Запуск Backend + zrok тунель ===
import subprocess, threading, time, re, os

ZROK_BIN = '/kaggle/working/zrok'
PUBLIC_URL = None

# 1. Запускаємо FastAPI
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'predator_app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    cwd='/kaggle/working'
)
time.sleep(4)

# Перевіряємо що сервер живий
import urllib.request
try:
    with urllib.request.urlopen('http://localhost:8000/api/v1/health', timeout=5) as r:
        print(f'✅ Backend ONLINE: {r.read().decode()[:80]}...')
except Exception as e:
    print(f'❌ Backend error: {e}')

# 2. Запускаємо zrok тунель
def run_tunnel():
    global PUBLIC_URL
    zrok_proc = subprocess.Popen(
        [ZROK_BIN, 'share', 'public', 'http://localhost:8000', '--headless'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in zrok_proc.stdout:
        print(f'[zrok] {line.rstrip()}')
        # Шукаємо URL у різних форматах
        match = re.search(r'https://[a-z0-9]+\.share\.zrok\.io', line)
        if not match:
            match = re.search(r'https://[\w\-]+\.zrok\.io[\S]*', line)
        if not match and 'access your share at' in line:
            parts = line.split('access your share at')
            if len(parts) > 1:
                PUBLIC_URL = parts[-1].strip()
        if match:
            PUBLIC_URL = match.group(0)
        if PUBLIC_URL:
            print('\n' + '='*60)
            print('🔥 PREDATOR KAGGLE NODE IS LIVE!')
            print(f'🔗 PUBLIC URL: {PUBLIC_URL}')
            print('='*60 + '\n')
            break

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()
t.join(timeout=30)  # чекаємо max 30 сек на URL

if not PUBLIC_URL:
    print('⚠️ URL ще не з\'явився — дивіться вивід [zrok] вище')

In [ ]:
# === КЛІТИНКА 6: Keep-Alive + Моніторинг ===
import time, psutil, urllib.request, json

print('🚀 PREDATOR Kaggle Node активний. Ctrl+C для зупинки.\n')
if PUBLIC_URL:
    print(f'🔗 URL: {PUBLIC_URL}')
    print(f'🔗 Health: {PUBLIC_URL}/api/v1/health')
    print(f'🔗 OODA:   {PUBLIC_URL}/api/v1/factory/ooda\n')

try:
    cycle = 0
    while True:
        cycle += 1
        mem = psutil.virtual_memory()
        cpu = psutil.cpu_percent(interval=1)
        
        # Перевіряємо health локально
        try:
            with urllib.request.urlopen('http://localhost:8000/api/v1/health', timeout=3) as r:
                data = json.loads(r.read())
                ooda_phase = 'N/A'
                status = data.get('status', '?')
        except Exception:
            status = 'ERROR'
        
        print(f'[{cycle:04d}] ✅ {status} | RAM: {mem.used/1024**3:.1f}/{mem.total/1024**3:.1f}GB ({mem.percent:.0f}%) | CPU: {cpu:.0f}%')
        time.sleep(30)
except KeyboardInterrupt:
    print('\n🛑 Зупинка...')